# SAM-WM frozen Kaggle benchmark

Run top-to-bottom. This notebook resolves one GitHub commit at bootstrap and records it. Do **not** use the retired `SAM_WM_V41_KAGGLE_INPUT.zip` workflow.

**Protocol boundary:** architecture/model selection may use Freiburg train + validation only. After the freeze-manifest cell is run, do not change code/config/preprocessing/QC for the reported run. Freiburg final test and both OOD evaluations are then opened once per seed.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

REPOSITORY = "AnnyaB/SAM-WM"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "SAM-WM"

def github_token() -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def github_json(url: str, token: str | None) -> dict:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        if exc.code in {401, 403, 404}:
            raise RuntimeError(
                "GitHub source is not accessible. If the repository is private, "
                "add a Kaggle Secret named GITHUB_TOKEN with read access to AnnyaB/SAM-WM."
            ) from exc
        raise

def download_archive(url: str, dst: Path, token: str | None) -> None:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=120) as response, dst.open("wb") as handle:
        shutil.copyfileobj(response, handle)

def safe_extract(archive: Path, dst: Path) -> Path:
    dst.mkdir(parents=True, exist_ok=True)
    root = dst.resolve()
    with tarfile.open(archive, mode="r:gz") as tf:
        members = tf.getmembers()
        for member in members:
            target = (root / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"unsafe archive path: {member.name}")
            if member.issym() or member.islnk():
                raise RuntimeError(f"archive links are not accepted: {member.name}")
        tf.extractall(root)
    directories = [path for path in root.iterdir() if path.is_dir()]
    if len(directories) != 1:
        raise RuntimeError(f"unexpected GitHub archive layout: {directories}")
    return directories[0]

token = github_token()
meta = github_json(f"https://api.github.com/repos/{REPOSITORY}/commits/main", token)
SOURCE_SHA = str(meta["sha"])
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA):
    raise RuntimeError("GitHub returned an invalid commit SHA")

if REPO.exists():
    shutil.rmtree(REPO)

with tempfile.TemporaryDirectory(prefix="samwm-source-") as tmp_name:
    tmp = Path(tmp_name)
    archive = tmp / "source.tar.gz"
    download_archive(
        f"https://api.github.com/repos/{REPOSITORY}/tarball/{SOURCE_SHA}",
        archive,
        token,
    )
    extracted = safe_extract(archive, tmp / "extract")
    shutil.copytree(extracted, REPO)

os.chdir(REPO)
(REPO / "artifacts").mkdir(exist_ok=True)
(REPO / "artifacts" / "FROZEN_SOURCE_SHA.txt").write_text(
    SOURCE_SHA + "\n", encoding="utf-8"
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
subprocess.run(["make", f"PYTHON={sys.executable}", "verify"], check=True)
print("Frozen source SHA:", SOURCE_SHA)


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))


## Train three independent seeds

The Freiburg loader downloads the official Zenodo files with registered MD5 checks when they are absent. No final-test or OOD metric is computed here.


In [ ]:
def run_python(*args: str) -> None:
    subprocess.run([sys.executable, *args], check=True)

for seed in (0, 1, 2):
    run_python("train.py", "--seed", str(seed), "--out", "artifacts/freiburg")


## Validation only — model selection may use these results


In [ ]:
for seed in (0, 1, 2):
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/freiburg/seed_{seed}/best.pt",
        "--data", "freiburg",
        "--split", "validation",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## Freeze the reported experiment

This writes the source/config/checkpoint hashes and verifies that all three validation artifacts exist. After this point, **do not alter the reported model or data protocol**.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

validation_files = [
    REPO / f"artifacts/eval/seed_{seed}/freiburg_validation_metrics.json"
    for seed in (0, 1, 2)
]
missing = [str(path) for path in validation_files if not path.exists()]
if missing:
    raise RuntimeError(f"Validation evidence is incomplete: {missing}")

freeze = {
    "source_sha": (REPO / "artifacts/FROZEN_SOURCE_SHA.txt").read_text(
        encoding="utf-8"
    ).strip(),
    "config_sha256": sha256_file(REPO / "config/train.yaml"),
    "checkpoints": {
        f"seed_{seed}": sha256_file(REPO / f"artifacts/freiburg/seed_{seed}/best.pt")
        for seed in (0, 1, 2)
    },
    "validation_artifacts": {
        f"seed_{seed}": sha256_file(validation_files[seed])
        for seed in (0, 1, 2)
    },
}
freeze_path = REPO / "artifacts/FREEZE_MANIFEST.json"
freeze_path.write_text(
    json.dumps(freeze, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(freeze_path.read_text(encoding="utf-8"))


## Freiburg final test — open once after freeze


In [ ]:
for seed in (0, 1, 2):
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/freiburg/seed_{seed}/best.pt",
        "--data", "freiburg",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-1 — Novi Sad zero-shot, no fine-tuning or OOD recalibration


In [ ]:
for seed in (0, 1, 2):
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/freiburg/seed_{seed}/best.pt",
        "--data", "novisad",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-2 — FAIRUrbTemp zero-shot

Before running this cell, attach the official DOI `10.48620/93247` extracted dataset as a Kaggle Input. Set `FAIR_ROOT` to that extracted root and set `FAIR_CITY` **before viewing any SAM-WM FAIRUrbTemp result**. The same city is used for all three seeds.


In [ ]:
FAIR_ROOT = Path("/kaggle/input/REPLACE_WITH_FAIRURBTEMP/extracted-root")
FAIR_CITY = "REPLACE_WITH_PREREGISTERED_CITY"

if "REPLACE_WITH" in str(FAIR_ROOT) or FAIR_CITY.startswith("REPLACE_WITH"):
    raise RuntimeError(
        "Set FAIR_ROOT and FAIR_CITY before opening FAIRUrbTemp held-out evaluation."
    )
if not FAIR_ROOT.exists():
    raise FileNotFoundError(FAIR_ROOT)

for seed in (0, 1, 2):
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/freiburg/seed_{seed}/best.pt",
        "--data", "fairurbtemp",
        "--root", str(FAIR_ROOT),
        "--city", FAIR_CITY,
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## Aggregate three-seed evidence and produce frozen figures


In [ ]:
run_python("summarize.py", "--root", "artifacts/eval", "--out", "artifacts/summary.json")
for seed in (0, 1, 2):
    run_python(
        "plot.py",
        f"artifacts/eval/seed_{seed}/freiburg_heldout_metrics.json",
        f"artifacts/eval/seed_{seed}/novisad_heldout_metrics.json",
        f"artifacts/eval/seed_{seed}/fairurbtemp_heldout_metrics.json",
        "--out", f"artifacts/figures/seed_{seed}",
    )

summary = Path("artifacts/summary.json")
print(summary.read_text(encoding="utf-8"))
